In [ ]:
from pathlib import Path
import numpy as np
import wfdb # type: ignore
import matplotlib.pyplot as plt
import pandas as pd
import csv
import pywt # type: ignore
from scipy.signal import hilbert
from scipy.signal import find_peaks
import ast

: 

In [ ]:
def strip_wfdb_suffix(p: Path) -> Path:

    while p.suffix in (".dat", ".hea"):
        p = p.with_suffix("")
    return p

In [ ]:
def detectar_picos_R(derivacion_I, wavelet, nivel):
    coeffs = pywt.wavedec(derivacion_I, wavelet, level=nivel)
    coeffs_qrs = [np.zeros_like(c) for c in coeffs]
    coeffs_qrs[4] = coeffs[4]  # d5
    coeffs_qrs[5] = coeffs[5]  # d4
    qrs_signal = pywt.waverec(coeffs_qrs, 'db6')
    qrs_signal = qrs_signal[:len(derivacion_I)]
    analytic_signal = hilbert(qrs_signal)
    envelope = np.abs(analytic_signal)
    umbral = np.mean(envelope) + 1.0 * np.std(envelope)
    peaks, _ = find_peaks(envelope, height=umbral, distance=200)
    return peaks

In [ ]:
def extraer_latidos(paciente,derivacion,pre,post):
    latidos_recortados=[]
    for latidos in paciente:
        if latidos-pre>=0 and latidos+post<len(derivacion):
            latido=derivacion[latidos-pre:latidos+post]
            latidos_recortados.append(latido)
    latidos_recortados=np.array(latidos_recortados)
    return latidos_recortados

def pintar_latidos(latidos, paciente):
    tiempo = np.linspace(-0.2, 0.4, pre+post)  # eje de tiempo
    
    plt.figure(figsize=(10,6))  # figura única por paciente

    for beat in latidos:
        plt.plot(tiempo, beat, alpha=0.6)  # todos los latidos en la misma gráfica

    plt.axvline(0, color='red', linestyle='--', label='R peak')
    plt.title("Latidos alineados en R del paciente {}".format(paciente))
    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud")
    plt.legend()
    plt.show()

In [ ]:
def construir_matriz_paciente(latidos,max_latidos,longitud_latido):
    matriz_paciente = np.zeros((max_latidos, longitud_latido))
    for i in range(min(len(latidos), max_latidos)):
        latido = latidos[i]
        
        # por seguridad si alguna ventana sale rara
        if len(latido) >= longitud_latido:
            matriz_paciente[i] = latido[:longitud_latido]
        else:
            matriz_paciente[i, :len(latido)] = latido
    return matriz_paciente

def construir_vector_etiqueta(etiqueta, enfermedades):
    etiqueta = ast.literal_eval(etiqueta)
    y_vector = np.zeros(len(enfermedades))
    
    for i, clase in enumerate(enfermedades):
        if clase in etiqueta:
            y_vector[i] = etiqueta[clase] / 100.0
    return y_vector

def vector_etiqueta_grupo(etiqueta, enfermedad_grupo,ENFERMEDADES):
    etiqueta = ast.literal_eval(etiqueta)
    y_vector = np.zeros(len(ENFERMEDADES))
    for enfermedad,prob in etiqueta.items():
        if prob>0:
            grupo=enfermedad_grupo.get(enfermedad, None)
            if grupo in ENFERMEDADES:
                idx = ENFERMEDADES.index(grupo)
                y_vector[idx] = 1
    return y_vector
    

In [ ]:
df_pacientes=pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\ptbxl_database.csv")
df_scp=pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\scp_statements.csv")

In [ ]:
df_pacientes

In [ ]:
df_pacientes = df_pacientes[(df_pacientes["age"] != df_pacientes["age"].max()) & (df_pacientes["age"] >= 18)]

In [ ]:
df_pacientes

In [ ]:
enfermedad_grupo=dict(zip(df_scp.iloc[:, 0], df_scp['diagnostic_class']))

In [ ]:
fs=500
pre=int(0.2*fs)#100 muestras antes del pico R
post=int(0.4*fs)#200 muestras después del pico R
MAX_LATIDOS = 10#Lo reducimos a 10 porque así hay más datos
LONGITUD_LATIDO = 300
wavelet='db6'
nivel=8
ENFERMEDADES=list(df_scp['diagnostic_class'].unique())
ENFERMEDADES=ENFERMEDADES[:-1] 

In [ ]:
carpeta=Path("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\Datos\\data")
#Datos 00001-00999:Faltan el 137,139,140,141,142,143,145,456,458,459,461,462
#Datos 01000-01999:No faltan datos 
#Datos 02000-02999:Faltan el 2506,2511
#Datos 03000-03999:Faltan el 3795,3798,3800,3801,3832
#Datos 04000-04999:No faltan datos
#Datos 05000-05999:Falta el 5817
#Datos 06000-06999:No faltan datos
#Datos 07000-07999:Falta el 07777, 07779,07782
#Datos 08000-08999:No faltan datos
#Datos 09000-09999:Falta el 9821,9825,9888
#Datos 10000-10999:No faltan datos
#Datos 11000-11999:Falta el 11810,11814,11815,11817,11838
#Datos 12000-12999:No faltan datos
#Datos 13000-13999:Faltan el 13791.13793,13796,13797,13799
#Datos 14000-14999:No faltan datos
#Datos 15000-15999:Falta el 15742
#Datos 16000-16999:No faltan datos
#Datos 17000-17999:No faltan datos (Los he vuelto a descargar porque daban error 56 de estos archivos)
#Datos 18000-18999:Falta el 18150
#Datos 19000-19999:No faltan datos
#Datos 20000-20999:No faltan datos
#Datos 21000-21999:Faltan el 21838-21999

In [ ]:
picos_R_pacientes    = []
latidos_pacientes    = []
latidos_12_pacientes = []
pacientes_validos    = []

for archivo in carpeta.glob("*.hea"):
    record       = archivo.stem
    BASE         = strip_wfdb_suffix(archivo)
    p_signal, _  = wfdb.rdsamp(str(BASE))
    p_signal_t   = p_signal.T
    derivacion_I = p_signal_t[0, :]

    # Detectamos picos R siempre sobre la derivación I
    Rloc    = detectar_picos_R(derivacion_I, wavelet, nivel)
    latidos = extraer_latidos(Rloc, derivacion_I, pre, post)

    if len(latidos) < MAX_LATIDOS:
        continue

    latidos         = latidos[:MAX_LATIDOS]
    matriz_paciente = construir_matriz_paciente(latidos, MAX_LATIDOS, LONGITUD_LATIDO)

    # Extraemos latidos de las 12 derivaciones con los mismos picos R
    latidos_12 = []
    for d in range(12):
        derivacion_d = p_signal_t[d, :]
        latidos_d    = extraer_latidos(Rloc, derivacion_d, pre, post)
        if len(latidos_d) < MAX_LATIDOS:
            matriz_d = np.zeros((MAX_LATIDOS, LONGITUD_LATIDO))
        else:
            latidos_d = latidos_d[:MAX_LATIDOS]
            matriz_d  = construir_matriz_paciente(latidos_d, MAX_LATIDOS, LONGITUD_LATIDO)
        latidos_12.append(matriz_d)
    latidos_12 = np.array(latidos_12)  # (12, MAX_LATIDOS, LONGITUD_LATIDO)

    picos_R_pacientes.append(Rloc)
    latidos_pacientes.append(matriz_paciente)
    latidos_12_pacientes.append(latidos_12)
    pacientes_validos.append(record)

print(f"Pacientes válidos: {len(pacientes_validos)}")

In [ ]:
print(latidos_12.shape())

In [ ]:
X      = []
Y      = []
pacientes_validos_final    = []
latidos_pacientes_final    = []
latidos_12_pacientes_final = []
picos_R_pacientes_final    = []

for i, record in enumerate(pacientes_validos):
    fila = df_pacientes.loc[
        df_pacientes['filename_hr'].str.contains(record, case=False, na=False)
    ]
    if len(fila) == 0:
        continue

    latidos    = latidos_pacientes[i]
    latidos_12 = latidos_12_pacientes[i]
    edad = fila['age'].values[0]
    sexo = 1.0 if fila['sex'].values[0] == 1 else 0.0

    # Megavector solo derivación I + edad y sexo
    megavector = latidos.flatten()
    X.append(np.append(megavector, [edad, sexo]))

    etiqueta = fila['scp_codes'].values[0]
    y = vector_etiqueta_grupo(etiqueta, enfermedad_grupo, ENFERMEDADES)
    Y.append(y)

    pacientes_validos_final.append(record)
    latidos_pacientes_final.append(latidos_pacientes[i])
    latidos_12_pacientes_final.append(latidos_12_pacientes[i])
    picos_R_pacientes_final.append(picos_R_pacientes[i])

X = np.array(X)
Y = np.array(Y)

print(f"Pacientes finales: {len(pacientes_validos_final)}")
print("Shape X:    ", X.shape)     # (N, 3302)
print("Shape Y:    ", Y.shape)

np.save('X_raw.npy',             X)
np.save('Y.npy',                 Y)
np.save('picos_R.npy',           np.array(picos_R_pacientes_final,    dtype=object), allow_pickle=True)
np.save('latidos.npy',           np.array(latidos_pacientes_final,    dtype=object), allow_pickle=True)
np.save('latidos_12.npy',        np.array(latidos_12_pacientes_final, dtype=object), allow_pickle=True)
np.save('pacientes_validos.npy', np.array(pacientes_validos_final),                  allow_pickle=True)
print("Guardado correctamente")